# Command Line Examples

## Get Interface

In [ ]:
import math
import socket
import sys
import threading
import urllib.parse
from http.server import BaseHTTPRequestHandler, HTTPServer

import epicsarchiver.retrieval.EPICSEvent_pb2 as _ee
from epicsarchiver.retrieval.EPICSEvent_pb2 import SCALAR_DOUBLE, PayloadInfo, ScalarDouble
from epicsarchiver.retrieval.pb import escape_bytes

_YEAR = 2026
_BASE_SECS = 8_640_000


def _make_doc_events(n=30):
    return [
        ScalarDouble(
            secondsintoyear=_BASE_SECS + i * 10,
            nano=int(abs(math.sin(i)) * 1_000_000_000),
            val=28.0 + 0.5 * math.sin(i * 0.4),
            severity=1,
            status=4,
            fieldvalues=[
                _ee.FieldValue(name="EGU", val="degC"),
                _ee.FieldValue(name="PREC", val="2"),
            ],
        )
        for i in range(n)
    ]


def _make_pb(pvname):
    events = _make_doc_events()
    info = PayloadInfo(type=SCALAR_DOUBLE, pvname=pvname, year=_YEAR)
    info_bytes = escape_bytes(info.SerializeToString())
    events_bytes = b"\n".join(escape_bytes(e.SerializeToString()) for e in events)
    return info_bytes + b"\n" + events_bytes


class _MockHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        parsed = urllib.parse.urlparse(self.path)
        qs = urllib.parse.parse_qs(parsed.query)
        pv = qs.get("pv", ["unknown"])[0]
        body = _make_pb(pv)
        self.send_response(200)
        self.send_header("Content-Type", "application/octet-stream")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *args):
        pass  # silence server logs


class _ReusableHTTPServer(HTTPServer):
    allow_reuse_address = True


_mock_server = _ReusableHTTPServer(("localhost", 17668), _MockHandler)
_server_thread = threading.Thread(target=_mock_server.serve_forever, daemon=True)
_server_thread.start()

In [ ]:
!arch-retrieval get --help

In [ ]:
!arch-retrieval --hostname localhost get EXAMPLE:TEMPERATURE

In [ ]:
!arch-retrieval --hostname localhost get EXAMPLE:TEMPERATURE -p min -b 5

In [ ]:
!arch-retrieval --hostname localhost get EXAMPLE:TEMPERATURE -p ncount

In [ ]:
!arch-retrieval --hostname localhost get EXAMPLE:TEMPERATURE2 EXAMPLE:TEMPERATURE3  -s 2024-12-01T12:00:00 -e 2024-12-16T12:10:10.10 -p MEDIAN -b 6000

In [ ]:
_mock_server.shutdown()